In [4]:
import os
import pandas as pd
import numpy as np
from wordfreq import word_frequency

# 1. Define paths relative to the notebooks directory
DATA_DIR = os.path.join("..", "data")
ALLOWED_GUESSES_PATH = os.path.join(DATA_DIR, "allowed_guesses.csv")
PRIORS_PATH = os.path.join(DATA_DIR, "unseen_cleaned_priors.csv")

# 2. Load allowed guesses (No header, single column)
allowed_df = pd.read_csv(ALLOWED_GUESSES_PATH, header=None, names=['word'])
allowed_guesses = allowed_df['word'].astype(str).str.strip().str.lower().tolist()

# 3. Load the unseen priors (Has headers: word, prior, expected_guesses)
priors_df = pd.read_csv(PRIORS_PATH)
priors_df['word'] = priors_df['word'].astype(str).str.strip().str.lower()
priors_df['prior'] = priors_df['prior'].astype(float)
priors_dict = dict(zip(priors_df['word'], priors_df['prior']))

# 4. Build the master DataFrame for all words
df = pd.DataFrame({'word': allowed_guesses})
df = df.drop_duplicates(subset=['word']).reset_index(drop=True)

# 5. Compute raw frequency using wordfreq (occurrences per word in corpus)
df['raw_frequency'] = df['word'].apply(lambda w: word_frequency(w, 'en'))

# 6. Calculate frequency rank (1 = most common)
df['freq_rank'] = df['raw_frequency'].rank(ascending=False, method='min').astype(int)

# 7. Map the unseen priors to the dataframe
df['unseen_prior'] = df['word'].map(lambda w: priors_dict.get(w, 0.0))

# Sort by frequency rank for quick inspection
df = df.sort_values(by='freq_rank').reset_index(drop=True)

# ---------------------------------------------------------------------
# DIAGNOSTIC: Check the non-zero prior words with the worst rankings
# ---------------------------------------------------------------------
# Filter for words that are plausible solutions (prior > 0)
non_zero_priors = df[df['unseen_prior'] > 1e-9].copy()

# Sort descending to find the solution words with the worst (highest) frequency ranks
worst_ranked_targets = non_zero_priors.sort_values(by='freq_rank', ascending=False).head(10)

print("=== 10 NON-ZERO PRIOR WORDS WITH WORST FREQUENCY RANKINGS ===")
print(worst_ranked_targets[['word', 'raw_frequency', 'freq_rank', 'unseen_prior']].to_string(index=False))

# Calculate how many target words drop below the 4,500 cutoff line
outside_cutoff = non_zero_priors[non_zero_priors['freq_rank'] > 4500]
print("\n-------------------------------------------------------------")
print(f"Total target words outside the top 4,500 cutoff: {len(outside_cutoff)} out of {len(non_zero_priors)}")

=== 10 NON-ZERO PRIOR WORDS WITH WORST FREQUENCY RANKINGS ===
 word  raw_frequency  freq_rank  unseen_prior
retag            0.0       9794      0.031973
aboil            0.0       9794      0.663523
laved            0.0       9794      0.003896
unarm            0.0       9794      0.052968
vined            0.0       9794      0.003896
meshy            0.0       9794      0.040901
befog            0.0       9794      0.736992
wiled            0.0       9794      0.003896
besot            0.0       9794      0.040901
moper            0.0       9794      0.736992

-------------------------------------------------------------
Total target words outside the top 4,500 cutoff: 540 out of 3209
